### Allscripts Sunrise (SCM) Device Exposure Hydration

Surgical material usage promoted into OMOP `device_exposure`.
This version aligns patient linkage to the Sunrise `cv3client` keyspace instead of the non-existent `rpt_ptnt_vw` source key used previously.

In [0]:
%sql
-- TRUNCATE TABLE _exponent.omop_allscripts.device_exposure;

In [0]:
%sql
TRUNCATE TABLE _exponent.omop_scm.device_exposure;


In [0]:
%sql
DELETE FROM _exponent.omop_mapping.source_to_device_exposure
WHERE source_system = 'allscripts_scm';

In [0]:
%sql
DELETE FROM _exponent.omop_silver.device_exposure
WHERE source_system = 'allscripts_scm';

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW device_exposure_silver AS
WITH staged AS (
  SELECT
    CONCAT_WS(CHR(31), 'allscripts_scm', 'rpt_case_mtrl_use_vw', 'MTRL_USE_FCT_ID', CAST(mu.MTRL_USE_FCT_ID AS STRING)) AS device_exposure_source_value,
    stp.person_id,
    COALESCE(dev_concept.omop_concept_id, 0) AS device_concept_id,
    CAST(mu.CASE_STRT_TS AS DATE) AS device_exposure_start_date,
    mu.CASE_STRT_TS AS device_exposure_start_datetime,
    CAST(COALESCE(mu.CASE_END_TS, mu.CASE_STRT_TS) AS DATE) AS device_exposure_end_date,
    COALESCE(mu.CASE_END_TS, mu.CASE_STRT_TS) AS device_exposure_end_datetime,
    32817 AS device_type_concept_id,
    COALESCE(m.MFC_PART_NUM, m.MFC_MODL_NUM) AS unique_device_id,
    m.VNDR_PART_NUM AS production_id,
    CAST(COALESCE(mu.MTRL_QTY_NUM, 1) AS DOUBLE) AS quantity,
    NULL AS provider_id,
    NULL AS visit_occurrence_id,
    NULL AS visit_detail_id,
    COALESCE(m.MTRL_NM, m.MTRL_DESC, m.PPLSOFT_ITM_NUM, m.VNDR_PART_NUM) AS device_source_value,
    0 AS device_source_concept_id,
    NULL AS unit_concept_id,
    m.UNIT_OF_ISSU_DESC AS unit_source_value,
    NULL AS unit_source_concept_id,
    'allscripts_scm' AS source_system,
    CURRENT_TIMESTAMP() AS last_mod_tsp
  FROM _exponent._bronze_srgry_dmart.rpt_case_mtrl_use_vw mu
  LEFT JOIN _exponent._bronze_srgry_dmart.rpt_mtrl_vw m
    ON m.MTRL_DIM_ID = mu.MTRL_DIM_ID
  INNER JOIN _exponent.omop_mapping.source_to_person stp
    ON stp.person_source_value = CONCAT_WS(
         CHR(31),
         'allscripts_scm',
         'cv3client',
         'GUID',
         REGEXP_EXTRACT(mu.MTRL_USE_BK, 'patient_id~(\\d+)', 1)
       )
   AND stp.active_flag = TRUE
  LEFT JOIN _exponent.omop_mapping.domain_source_to_concept dev_concept
    ON dev_concept.domain_id = 'Device'
   AND dev_concept.source_system = 'allscripts_scm'
   AND dev_concept.source_id = COALESCE(m.VNDR_PART_NUM, m.MFC_PART_NUM, m.PPLSOFT_ITM_NUM, m.MTRL_NM)
  WHERE mu.MTRL_USE_FCT_ID IS NOT NULL
    AND mu.MTRL_USE_BK IS NOT NULL
    AND REGEXP_EXTRACT(mu.MTRL_USE_BK, 'patient_id~(\\d+)', 1) <> ''
    AND mu.CASE_STRT_TS IS NOT NULL
)
SELECT * FROM staged;

In [0]:
%sql
MERGE INTO _exponent.omop_silver.device_exposure AS t
USING device_exposure_silver AS s
ON t.device_exposure_source_value = s.device_exposure_source_value

WHEN MATCHED AND NOT (
     t.person_id <=> s.person_id
 AND t.device_concept_id <=> s.device_concept_id
 AND t.device_exposure_start_date <=> s.device_exposure_start_date
 AND t.device_exposure_start_datetime <=> s.device_exposure_start_datetime
 AND t.device_exposure_end_date <=> s.device_exposure_end_date
 AND t.device_exposure_end_datetime <=> s.device_exposure_end_datetime
 AND t.unique_device_id <=> s.unique_device_id
 AND t.production_id <=> s.production_id
 AND t.quantity <=> s.quantity
 AND t.device_source_value <=> s.device_source_value
 AND t.unit_source_value <=> s.unit_source_value
 AND t.source_system <=> s.source_system
)
THEN UPDATE SET
  t.person_id = s.person_id,
  t.device_concept_id = s.device_concept_id,
  t.device_exposure_start_date = s.device_exposure_start_date,
  t.device_exposure_start_datetime = s.device_exposure_start_datetime,
  t.device_exposure_end_date = s.device_exposure_end_date,
  t.device_exposure_end_datetime = s.device_exposure_end_datetime,
  t.device_type_concept_id = s.device_type_concept_id,
  t.unique_device_id = s.unique_device_id,
  t.production_id = s.production_id,
  t.quantity = s.quantity,
  t.provider_id = s.provider_id,
  t.visit_occurrence_id = s.visit_occurrence_id,
  t.visit_detail_id = s.visit_detail_id,
  t.device_source_value = s.device_source_value,
  t.device_source_concept_id = s.device_source_concept_id,
  t.unit_concept_id = s.unit_concept_id,
  t.unit_source_value = s.unit_source_value,
  t.unit_source_concept_id = s.unit_source_concept_id,
  t.source_system = s.source_system,
  t.last_mod_tsp = s.last_mod_tsp

WHEN NOT MATCHED THEN INSERT (
  device_exposure_source_value,
  person_id,
  device_concept_id,
  device_exposure_start_date,
  device_exposure_start_datetime,
  device_exposure_end_date,
  device_exposure_end_datetime,
  device_type_concept_id,
  unique_device_id,
  production_id,
  quantity,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  device_source_value,
  device_source_concept_id,
  unit_concept_id,
  unit_source_value,
  unit_source_concept_id,
  source_system,
  last_mod_tsp
)
VALUES (
  s.device_exposure_source_value,
  s.person_id,
  s.device_concept_id,
  s.device_exposure_start_date,
  s.device_exposure_start_datetime,
  s.device_exposure_end_date,
  s.device_exposure_end_datetime,
  s.device_type_concept_id,
  s.unique_device_id,
  s.production_id,
  s.quantity,
  s.provider_id,
  s.visit_occurrence_id,
  s.visit_detail_id,
  s.device_source_value,
  s.device_source_concept_id,
  s.unit_concept_id,
  s.unit_source_value,
  s.unit_source_concept_id,
  s.source_system,
  s.last_mod_tsp
);

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_device_exposure (
  source_system,
  device_exposure_source_value,
  active_flag,
  created_tsp,
  last_mod_tsp,
  merge_id,
  merge_reason
)
SELECT
  s.source_system,
  s.device_exposure_source_value,
  TRUE,
  CURRENT_TIMESTAMP(),
  COALESCE(s.last_mod_tsp, CURRENT_TIMESTAMP()),
  NULL,
  NULL
FROM _exponent.omop_silver.device_exposure s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_device_exposure x
  ON s.device_exposure_source_value = x.device_exposure_source_value
WHERE s.source_system = 'allscripts_scm';

In [0]:
%sql
MERGE INTO _exponent.omop_scm.device_exposure AS gold
USING (
  SELECT
    std.device_exposure_id,
    s.person_id,
    s.device_concept_id,
    s.device_exposure_start_date,
    s.device_exposure_start_datetime,
    s.device_exposure_end_date,
    s.device_exposure_end_datetime,
    s.device_type_concept_id,
    s.unique_device_id,
    s.production_id,
    s.quantity,
    s.provider_id,
    s.visit_occurrence_id,
    s.visit_detail_id,
    s.device_source_value,
    s.device_source_concept_id,
    s.unit_concept_id,
    s.unit_source_value,
    s.unit_source_concept_id
  FROM _exponent.omop_silver.device_exposure s
  JOIN _exponent.omop_mapping.source_to_device_exposure std
    ON std.device_exposure_source_value = s.device_exposure_source_value
   AND std.active_flag = TRUE
  WHERE s.source_system = 'allscripts_scm'
) AS src
ON gold.device_exposure_id = src.device_exposure_id

WHEN MATCHED THEN UPDATE SET
  gold.person_id = src.person_id,
  gold.device_concept_id = src.device_concept_id,
  gold.device_exposure_start_date = src.device_exposure_start_date,
  gold.device_exposure_start_datetime = src.device_exposure_start_datetime,
  gold.device_exposure_end_date = src.device_exposure_end_date,
  gold.device_exposure_end_datetime = src.device_exposure_end_datetime,
  gold.device_type_concept_id = src.device_type_concept_id,
  gold.unique_device_id = src.unique_device_id,
  gold.production_id = src.production_id,
  gold.quantity = src.quantity,
  gold.provider_id = src.provider_id,
  gold.visit_occurrence_id = src.visit_occurrence_id,
  gold.visit_detail_id = src.visit_detail_id,
  gold.device_source_value = src.device_source_value,
  gold.device_source_concept_id = src.device_source_concept_id,
  gold.unit_concept_id = src.unit_concept_id,
  gold.unit_source_value = src.unit_source_value,
  gold.unit_source_concept_id = src.unit_source_concept_id

WHEN NOT MATCHED THEN INSERT (
  device_exposure_id,
  person_id,
  device_concept_id,
  device_exposure_start_date,
  device_exposure_start_datetime,
  device_exposure_end_date,
  device_exposure_end_datetime,
  device_type_concept_id,
  unique_device_id,
  production_id,
  quantity,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  device_source_value,
  device_source_concept_id,
  unit_concept_id,
  unit_source_value,
  unit_source_concept_id
)
VALUES (
  src.device_exposure_id,
  src.person_id,
  src.device_concept_id,
  src.device_exposure_start_date,
  src.device_exposure_start_datetime,
  src.device_exposure_end_date,
  src.device_exposure_end_datetime,
  src.device_type_concept_id,
  src.unique_device_id,
  src.production_id,
  src.quantity,
  src.provider_id,
  src.visit_occurrence_id,
  src.visit_detail_id,
  src.device_source_value,
  src.device_source_concept_id,
  src.unit_concept_id,
  src.unit_source_value,
  src.unit_source_concept_id
);